# Fase 3 · M09: Perfiles de Riesgo

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M09 — Perfiles de Riesgo |

---

## 🎯 Qué hace

Construye perfiles de riesgo de abandono a partir de `df_eda_final.parquet`.
Segmenta alumnos en tres niveles (alto/medio/bajo riesgo) según variables clave
y genera análisis por rama, vía de acceso, beca y orden de preferencia.

## 📋 Requisitos

- `data/04_eda/df_eda_final.parquet` — generado por M08

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/04_eda/df_eda_perfiles.parquet` | Dataset con columna `perfil_riesgo` |
| `docs/html/fase3/m09_perfiles_riesgo.html` | Informe HTML con perfiles |

## 🔄 Flujo

```
df_eda_final.parquet (M08)
    ↓ Calcular score de riesgo por alumno
    ↓ Segmentar en alto/medio/bajo
    ↓ Análisis por rama, vía acceso, beca, preferencia
    → df_eda_perfiles.parquet + HTML
```

## ➡️ Siguiente

Fin de Fase 3. Continuar con Fase 4 (EDA Final).


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64, COLORES
from src.html import (
    generar_kpis_html, generar_seccion_html,
    generar_html_navegacion_completa, guardar_html
)
from src.html.render import render_pagina_desde_fichero

RUTA_EDA   = ROOT / 'data' / '04_eda'
RUTA_FASE3 = RUTA_HTML / 'fase3'
crear_directorios([RUTA_EDA, RUTA_FASE3])
fmt = formato_numero_es

info_entorno()
print('✓ Configuración completada')


✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR df_eda_final
# ============================================================================
# Fuente: M08 — dataset auditado con texto legible y titulacion incluida
# ============================================================================

print('=' * 60)
print('F3-M09: PERFILES DE RIESGO')
print('=' * 60)

df = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')
TARGET = 'abandono'

n_total   = len(df)
n_abandona = (df[TARGET] == 1).sum()
tasa_global = n_abandona / n_total

print(f'\n✓ Cargado: {fmt(n_total)} expedientes × {df.shape[1]} cols')
print(f'  Abandono: {fmt(n_abandona)} ({tasa_global*100:.1f}%)')
print(f'  Columnas: {df.columns.tolist()}')


F3-M09: PERFILES DE RIESGO

✓ Cargado: 33.621 expedientes × 20 cols
  Abandono: 9.833 (29.2%)
  Columnas: ['anios_gap', 'anios_sin_beca', 'cred_superados_anio_1er', 'edad_entrada', 'indicador_interrupcion', 'max_pagos', 'n_anios_beca', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'orden_preferencia', 'pais_nombre', 'provincia', 'rama', 'sexo', 'situacion_laboral', 'universidad_origen', 'via_acceso', 'abandono', 'titulacion']


In [3]:
# ============================================================================
# CELDA 3: SCORE DE RIESGO Y SEGMENTACIÓN
# ============================================================================
# Score basado en variables clave disponibles en df_eda_final.
# Cada variable aporta puntos según su asociación con el abandono.
# Score final: 0-100. Segmentación en terciles.
#
# Variables usadas (todas disponibles antes del abandono, sin leakage):
#   - nota_1er_anio: nota baja → más riesgo
#   - cred_superados_anio_1er: pocos créditos → más riesgo
#   - n_anios_beca: sin beca → más riesgo económico
#   - orden_preferencia: mayor número (o 0) → menos motivación
#   - indicador_interrupcion: interrupción → más riesgo
#   - tasa_abandono_titulacion: titulación con alta tasa histórica
# ============================================================================

print('=' * 60)
print('CONSTRUYENDO SCORE DE RIESGO')
print('=' * 60)

df_score = df.copy()

# --- Normalizar variables a escala 0-1 (0=bajo riesgo, 1=alto riesgo) ---

# nota_1er_anio: nota baja → riesgo alto
if 'nota_1er_anio' in df_score.columns:
    nota_max = df_score['nota_1er_anio'].quantile(0.95)
    nota_min = df_score['nota_1er_anio'].quantile(0.05)
    df_score['r_nota'] = 1 - ((df_score['nota_1er_anio'].clip(nota_min, nota_max) - nota_min) / (nota_max - nota_min + 1e-9))
else:
    df_score['r_nota'] = 0.5

# cred_superados_anio_1er: pocos créditos → riesgo alto
if 'cred_superados_anio_1er' in df_score.columns:
    cred_max = df_score['cred_superados_anio_1er'].quantile(0.95)
    cred_min = df_score['cred_superados_anio_1er'].quantile(0.05)
    df_score['r_cred'] = 1 - ((df_score['cred_superados_anio_1er'].clip(cred_min, cred_max) - cred_min) / (cred_max - cred_min + 1e-9))
else:
    df_score['r_cred'] = 0.5

# n_anios_beca: sin beca → riesgo alto
if 'n_anios_beca' in df_score.columns:
    beca_max = df_score['n_anios_beca'].max()
    df_score['r_beca'] = 1 - (df_score['n_anios_beca'] / (beca_max + 1e-9))
else:
    df_score['r_beca'] = 0.5

# indicador_interrupcion: interrupción → riesgo alto
if 'indicador_interrupcion' in df_score.columns:
    df_score['r_interrupcion'] = df_score['indicador_interrupcion'].astype(float)
else:
    df_score['r_interrupcion'] = 0.0

# tasa_abandono_titulacion: alta tasa → riesgo alto
if 'tasa_abandono_titulacion' in df_score.columns:
    df_score['r_titulacion'] = df_score['tasa_abandono_titulacion']
else:
    df_score['r_titulacion'] = tasa_global

# orden_preferencia: 0 (sin preinscripción) o número alto → más riesgo
if 'orden_preferencia' in df_score.columns:
    # 1 = primera opción (menos riesgo), 0 o >5 = más riesgo
    df_score['r_preferencia'] = df_score['orden_preferencia'].apply(
        lambda x: 0.2 if x == 1 else (0.5 if 2 <= x <= 3 else 0.8)
    )
else:
    df_score['r_preferencia'] = 0.5


# cred_repetidos: muchos créditos repetidos → riesgo alto
if 'cred_repetidos' in df_score.columns:
    cred_rep_max = df_score['cred_repetidos'].quantile(0.95)
    df_score['r_cred_rep'] = (df_score['cred_repetidos'].clip(0, cred_rep_max) / (cred_rep_max + 1e-9))
else:
    df_score['r_cred_rep'] = 0.0

# n_anios_sin_notas: años sin notas → riesgo alto
if 'n_anios_sin_notas' in df_score.columns:
    notas_max = df_score['n_anios_sin_notas'].quantile(0.95)
    df_score['r_sin_notas'] = (df_score['n_anios_sin_notas'].clip(0, notas_max) / (notas_max + 1e-9))
else:
    df_score['r_sin_notas'] = 0.0

# n_anios_trabajando: más años trabajando → más riesgo
if 'n_anios_trabajando' in df_score.columns:
    trab_max = df_score['n_anios_trabajando'].quantile(0.95)
    df_score['r_trabajando'] = (df_score['n_anios_trabajando'].clip(0, trab_max) / (trab_max + 1e-9))
else:
    df_score['r_trabajando'] = 0.0

# --- Score ponderado (pesos basados en literatura de deserción) ---
PESOS = {
    'r_nota':         0.25,  # rendimiento académico primer año — mayor predictor
    'r_cred':         0.20,  # créditos primer año — complementa nota
    'r_beca':         0.12,  # situación económica
    'r_titulacion':   0.12,  # contexto de la titulación
    'r_cred_rep':     0.10,  # créditos repetidos — dificultad acumulada
    'r_sin_notas':    0.08,  # años sin notas — desenganche académico
    'r_trabajando':   0.08,  # años trabajando — conflicto estudios/trabajo
    'r_preferencia':  0.03,  # motivación de elección
    'r_interrupcion': 0.02,  # interrupción en trayectoria
}

df_score['score_riesgo'] = sum(
    df_score[var] * peso for var, peso in PESOS.items()
) * 100  # escala 0-100

df_score['score_riesgo'] = df_score['score_riesgo'].round(1)

# --- Segmentación en terciles ---
q33 = df_score['score_riesgo'].quantile(0.33)
q67 = df_score['score_riesgo'].quantile(0.67)

def asignar_perfil(score):
    if score <= q33:  return 'Bajo'
    elif score <= q67: return 'Medio'
    else:              return 'Alto'

df_score['perfil_riesgo'] = df_score['score_riesgo'].apply(asignar_perfil)

# --- Resumen ---
print(f'\n📊 Score de riesgo:')
print(f'   Media:  {df_score["score_riesgo"].mean():.1f}')
print(f'   Umbral Bajo/Medio: {q33:.1f}')
print(f'   Umbral Medio/Alto: {q67:.1f}')
print(f'\n📊 Distribución de perfiles:')
for perfil in ['Bajo', 'Medio', 'Alto']:
    n = (df_score['perfil_riesgo'] == perfil).sum()
    tasa = df_score[df_score['perfil_riesgo'] == perfil][TARGET].mean()
    print(f'   {perfil:5s}: {fmt(n)} alumnos ({n/n_total*100:.1f}%) | abandono real: {tasa*100:.1f}%')


CONSTRUYENDO SCORE DE RIESGO

📊 Score de riesgo:
   Media:  52.8
   Umbral Bajo/Medio: 46.5
   Umbral Medio/Alto: 58.3

📊 Distribución de perfiles:
   Bajo : 10.325 alumnos (30.7%) | abandono real: 7.5%
   Medio: 10.684 alumnos (31.8%) | abandono real: 18.7%


   Alto : 12.612 alumnos (37.5%) | abandono real: 55.9%


In [4]:
# ============================================================================
# CELDA 4: ANÁLISIS POR DIMENSIONES
# ============================================================================
# Tasa de abandono real por perfil cruzado con variables clave.
# Permite identificar qué combinaciones tienen más riesgo.
# ============================================================================

print('=' * 60)
print('ANÁLISIS POR DIMENSIONES')
print('=' * 60)

resumen_dimensiones = {}

# Por rama
if 'rama' in df_score.columns:
    resumen_dimensiones['rama'] = (
        df_score.groupby('rama')[TARGET]
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'tasa_abandono', 'count': 'n_alumnos'})
        .sort_values('tasa_abandono', ascending=False)
        .round(3)
    )
    print(f'\n📊 Por rama:')
    print(resumen_dimensiones['rama'].to_string())

# Por perfil de riesgo
resumen_dimensiones['perfil'] = (
    df_score.groupby('perfil_riesgo').agg(
        n_alumnos=(TARGET, 'count'),
        tasa_abandono=(TARGET, 'mean'),
        score_medio=('score_riesgo', 'mean')
    ).round(3)
)
print(f'\n📊 Por perfil de riesgo:')
print(resumen_dimensiones['perfil'].to_string())

# Por vía de acceso
if 'via_acceso' in df_score.columns:
    resumen_dimensiones['via_acceso'] = (
        df_score.groupby('via_acceso')[TARGET]
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'tasa_abandono', 'count': 'n_alumnos'})
        .sort_values('tasa_abandono', ascending=False)
        .round(3)
    )
    print(f'\n📊 Por vía de acceso (top 5):')
    print(resumen_dimensiones['via_acceso'].head().to_string())

# Por beca
if 'n_anios_beca' in df_score.columns:
    df_score['tuvo_beca_label'] = df_score['n_anios_beca'].apply(
        lambda x: 'Con beca' if x > 0 else 'Sin beca'
    )
    resumen_dimensiones['beca'] = (
        df_score.groupby('tuvo_beca_label')[TARGET]
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'tasa_abandono', 'count': 'n_alumnos'})
        .round(3)
    )
    print(f'\n📊 Por beca:')
    print(resumen_dimensiones['beca'].to_string())


ANÁLISIS POR DIMENSIONES

📊 Por rama:
      tasa_abandono  n_alumnos
rama                          
TE            0.374       7600
EX            0.334        787
SO            0.289      18594
HU            0.260       2903
SA            0.159       3737

📊 Por perfil de riesgo:
               n_alumnos  tasa_abandono  score_medio
perfil_riesgo                                       
Alto               12612          0.559       66.638
Bajo               10325          0.075       39.524
Medio              10684          0.187       52.307

📊 Por vía de acceso (top 5):
                                tasa_abandono  n_alumnos
via_acceso                                              
Pruebas acceso mayores 40 años          0.577         52
Pruebas acceso mayores 45 años          0.566         53
Adaptación a Grado                      0.535       1553
Pruebas acceso mayores 25 años          0.460        642
Extranjeros CEE                         0.376        394

📊 Por beca:
             

In [5]:
# ============================================================================
# CELDA 5: GRÁFICOS
# ============================================================================

print('=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

graficos = {}
COLORES_PERFIL = {'Bajo': '#38a169', 'Medio': '#ECC94B', 'Alto': '#e53e3e'}

# G1: Distribución de perfiles
fig, ax = plt.subplots(figsize=(6, 4))
perfiles = ['Bajo', 'Medio', 'Alto']
counts = [( df_score['perfil_riesgo'] == p).sum() for p in perfiles]
bars = ax.bar(perfiles, counts, color=[COLORES_PERFIL[p] for p in perfiles], edgecolor='white')
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{fmt(n)}\n({n/n_total*100:.1f}%)', ha='center', fontsize=10)
ax.set_title('Distribución de Perfiles de Riesgo', fontweight='bold')
ax.set_ylabel('Número de alumnos')
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
graficos['distribucion'] = figura_a_base64(fig)
plt.close()

# G2: Tasa de abandono real por perfil
fig, ax = plt.subplots(figsize=(6, 4))
tasas = [df_score[df_score['perfil_riesgo']==p][TARGET].mean()*100 for p in perfiles]
bars = ax.bar(perfiles, tasas, color=[COLORES_PERFIL[p] for p in perfiles], edgecolor='white')
ax.axhline(tasa_global*100, color='#2d3748', linestyle='--', linewidth=1.5,
           label=f'Media global: {tasa_global*100:.1f}%')
for bar, t in zip(bars, tasas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{t:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Tasa de Abandono Real por Perfil', fontweight='bold')
ax.set_ylabel('% Abandono')
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
graficos['tasa_perfil'] = figura_a_base64(fig)
plt.close()

# G3: Tasa de abandono por rama
if 'rama' in df_score.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    df_rama = resumen_dimensiones['rama'].sort_values('tasa_abandono')
    colors_rama = ['#e53e3e' if t > tasa_global else '#38a169' for t in df_rama['tasa_abandono']]
    ax.barh(df_rama.index, df_rama['tasa_abandono']*100, color=colors_rama, edgecolor='white')
    ax.axvline(tasa_global*100, color='#2d3748', linestyle='--', linewidth=1.5)
    for i, (rama, row) in enumerate(df_rama.iterrows()):
        ax.text(row['tasa_abandono']*100 + 0.5, i, f'{row["tasa_abandono"]*100:.1f}%', va='center')
    ax.set_title('Tasa de Abandono por Rama', fontweight='bold')
    ax.set_xlabel('% Abandono')
    plt.tight_layout()
    graficos['rama'] = figura_a_base64(fig)
    plt.close()

# G4: Score de riesgo — distribución
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_score[df_score[TARGET]==0]['score_riesgo'], bins=30, alpha=0.6,
        color='#38a169', label='No abandona', density=True)
ax.hist(df_score[df_score[TARGET]==1]['score_riesgo'], bins=30, alpha=0.6,
        color='#e53e3e', label='Abandona', density=True)
ax.axvline(q33, color='#ECC94B', linestyle='--', linewidth=1.5, label=f'Umbral bajo/medio ({q33:.0f})')
ax.axvline(q67, color='#ed8936', linestyle='--', linewidth=1.5, label=f'Umbral medio/alto ({q67:.0f})')
ax.set_title('Distribución del Score de Riesgo por Resultado', fontweight='bold')
ax.set_xlabel('Score de Riesgo (0-100)')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
graficos['score_dist'] = figura_a_base64(fig)
plt.close()

print(f'✅ {len(graficos)} gráficos generados')


GENERANDO GRÁFICOS


✅ 4 gráficos generados


In [6]:
# ============================================================================
# CELDA 6: GUARDAR DATASET CON PERFILES
# ============================================================================
# Añade score_riesgo y perfil_riesgo al df_eda_final y guarda.
# No modifica df_eda_final original — genera un fichero separado.
# ============================================================================

cols_nuevas = ['score_riesgo', 'perfil_riesgo']
df_perfiles = df.copy()
df_perfiles['score_riesgo'] = df_score['score_riesgo']
df_perfiles['perfil_riesgo'] = df_score['perfil_riesgo']

ruta_perfiles = RUTA_EDA / 'df_eda_perfiles.parquet'
df_perfiles.to_parquet(ruta_perfiles, index=False)

print(f'💾 Guardado: {ruta_perfiles.name}')
print(f'   Filas: {len(df_perfiles):,}')
print(f'   Columnas: {df_perfiles.shape[1]} (original + score_riesgo + perfil_riesgo)')
print(f'\n   Perfiles:')
for p in ['Bajo', 'Medio', 'Alto']:
    n = (df_perfiles['perfil_riesgo']==p).sum()
    print(f'   {p}: {fmt(n)} ({n/len(df_perfiles)*100:.1f}%)')


💾 Guardado: df_eda_perfiles.parquet
   Filas: 33,621
   Columnas: 22 (original + score_riesgo + perfil_riesgo)

   Perfiles:
   Bajo: 10.325 (30.7%)
   Medio: 10.684 (31.8%)
   Alto: 12.612 (37.5%)


In [7]:
# ============================================================================
# CELDA 7: GENERAR HTML
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase3', modulo_activo='m09'
)

# KPIs
n_alto  = (df_score['perfil_riesgo'] == 'Alto').sum()
n_medio = (df_score['perfil_riesgo'] == 'Medio').sum()
n_bajo  = (df_score['perfil_riesgo'] == 'Bajo').sum()
tasa_alto = df_score[df_score['perfil_riesgo']=='Alto'][TARGET].mean()

KPIS = [
    {'valor': fmt(n_alto),            'titulo': 'Riesgo Alto'},
    {'valor': fmt(n_medio),           'titulo': 'Riesgo Medio'},
    {'valor': fmt(n_bajo),            'titulo': 'Riesgo Bajo'},
    {'valor': f'{tasa_alto*100:.1f}%','titulo': 'Abandono en Alto'},
]

# S1: Score y metodología
# Tabla de pesos generada dinámicamente desde PESOS (celda 3)
_DESC_VARS = {
    'r_nota':         ('nota_1er_anio',           'Rendimiento académico primer año'),
    'r_cred':         ('cred_superados_anio_1er',  'Créditos aprobados primer año'),
    'r_beca':         ('n_anios_beca',             'Apoyo económico (beca)'),
    'r_titulacion':   ('tasa_abandono_titulacion', 'Tasa histórica de la titulación'),
    'r_cred_rep':     ('cred_repetidos',           'Créditos repetidos — dificultad acumulada'),
    'r_sin_notas':    ('n_anios_sin_notas',        'Años sin notas — desenganche académico'),
    'r_trabajando':   ('n_anios_trabajando',       'Años trabajando — conflicto estudios/trabajo'),
    'r_preferencia':  ('orden_preferencia',        'Motivación de elección de carrera'),
    'r_interrupcion': ('indicador_interrupcion',   'Interrupción en trayectoria'),
}
pesos_html = ''.join([
    f'<tr><td style="padding:6px;font-family:monospace">{_DESC_VARS[var][0]}</td>'
    f'<td style="text-align:center">{int(peso*100)}%</td>'
    f'<td>{_DESC_VARS[var][1]}</td></tr>'
    for var, peso in PESOS.items()
    if var in _DESC_VARS
])

s_metodologia = generar_seccion_html('Metodología del Score', f'''
<p>Score de riesgo ponderado (0-100) basado en 6 variables clave disponibles
antes del abandono. Segmentación en terciles: Bajo / Medio / Alto.</p>
<table style="width:100%;border-collapse:collapse;margin-top:15px">
<tr style="background:#3182ce;color:white">
  <th style="padding:8px">Variable</th>
  <th style="text-align:center">Peso</th>
  <th>Justificación</th>
</tr>
{pesos_html}
</table>''', '⚙️')

# S2: Gráficos de perfiles
s_graficos = generar_seccion_html('Distribución y Validación', f'''
<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px">
  <div><img src="data:image/png;base64,{graficos['distribucion']}" style="max-width:100%"></div>
  <div><img src="data:image/png;base64,{graficos['tasa_perfil']}" style="max-width:100%"></div>
</div>
<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-top:20px">
  <div><img src="data:image/png;base64,{graficos.get('rama', '')}" style="max-width:100%"></div>
  <div><img src="data:image/png;base64,{graficos['score_dist']}" style="max-width:100%"></div>
</div>''', '📊')

# S3: Tabla resumen por perfil
filas_perfil = ''
for perfil in ['Alto', 'Medio', 'Bajo']:
    sub = df_score[df_score['perfil_riesgo']==perfil]
    color = '#e53e3e' if perfil=='Alto' else '#ECC94B' if perfil=='Medio' else '#38a169'
    filas_perfil += (
        f'<tr><td style="padding:8px;font-weight:bold;color:{color}">{perfil}</td>'
        f'<td style="text-align:center">{fmt(len(sub))}</td>'
        f'<td style="text-align:center">{len(sub)/n_total*100:.1f}%</td>'
        f'<td style="text-align:center;font-weight:bold">{sub[TARGET].mean()*100:.1f}%</td>'
        f'<td style="text-align:center">{sub["score_riesgo"].mean():.1f}</td></tr>'
    )

s_tabla = generar_seccion_html('Resumen por Perfil', f'''
<table style="width:100%;border-collapse:collapse">
<tr style="background:#2d3748;color:white">
  <th style="padding:10px">Perfil</th>
  <th style="text-align:center">Alumnos</th>
  <th style="text-align:center">% Total</th>
  <th style="text-align:center">Abandono Real</th>
  <th style="text-align:center">Score Medio</th>
</tr>
{filas_perfil}
</table>
<p style="margin-top:15px;color:#4a5568;font-size:13px">
  Validación: el perfil Alto tiene una tasa de abandono real significativamente
  superior a la media global ({tasa_global*100:.1f}%), lo que confirma la
  capacidad discriminativa del score.
</p>''', '📋')

contenido_html = generar_kpis_html(KPIS) + s_metodologia + s_graficos + s_tabla

html_completo = render_pagina_desde_fichero(
    'f3_m09_perfiles_riesgo.ipynb',
    contenido_html,
    carpeta_notebook='fase3_features'
)

ruta_html = RUTA_FASE3 / 'm09_perfiles_riesgo.html'
guardar_html(html_completo, ruta_html)
print(f'\n✅ HTML: {ruta_html}')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m09_perfiles_riesgo.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m09_perfiles_riesgo.html


In [8]:
# ============================================================================
# CELDA 8: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F3-M09 COMPLETADO — PERFILES DE RIESGO')
print('=' * 60)
print(f'\n📊 Perfiles generados:')
for p in ['Alto', 'Medio', 'Bajo']:
    n = (df_score['perfil_riesgo']==p).sum()
    tasa = df_score[df_score['perfil_riesgo']==p][TARGET].mean()
    print(f'   {p:5s}: {fmt(n)} alumnos | abandono real: {tasa*100:.1f}%')
print(f'\n💾 Ficheros generados:')
print(f'   data/04_eda/df_eda_perfiles.parquet')
print(f'   docs/html/fase3/m09_perfiles_riesgo.html')
print(f'\n🎯 Fase 3 cerrada. Siguiente: Fase 4 (EDA Final)')
print('=' * 60)



✅ F3-M09 COMPLETADO — PERFILES DE RIESGO

📊 Perfiles generados:
   Alto : 12.612 alumnos | abandono real: 55.9%
   Medio: 10.684 alumnos | abandono real: 18.7%
   Bajo : 10.325 alumnos | abandono real: 7.5%

💾 Ficheros generados:
   data/04_eda/df_eda_perfiles.parquet
   docs/html/fase3/m09_perfiles_riesgo.html

🎯 Fase 3 cerrada. Siguiente: Fase 4 (EDA Final)
